In [1]:
from tritonclient.http import InferInput, InferRequestedOutput, InferenceServerClient
import tritonclient.utils as trutils
import numpy as np

In [2]:
triton_client = InferenceServerClient(url="localhost:8900")

image_paths = [
    "/images/cat1.jpg",
    "/images/cat2.jpg",
    "/images/cat3.jpg"
]
image_paths = np.array(image_paths, dtype=object)

input_images_path = InferInput(
    name="IMAGE_PATH",
    shape=image_paths.shape,
    datatype=trutils.np_to_triton_dtype(image_paths.dtype),
)

input_images_path.set_data_from_numpy(image_paths)


res = triton_client.infer(
    "image-preprocessor",
    [input_images_path],
    outputs=[
        InferRequestedOutput("PREPROCESSED_IMAGE"),
    ],
)

In [3]:
a = res.as_numpy("PREPROCESSED_IMAGE")
print(a.shape)

(3, 3, 96, 96)


In [4]:
triton_client = InferenceServerClient(url="localhost:8900")

preproc_input = InferInput(
    name="PREPROCESSED_IMAGE",
    shape=a.shape,
    datatype=trutils.np_to_triton_dtype(a.dtype),
)

preproc_input.set_data_from_numpy(a)


res = triton_client.infer(
    "conv-encoder-FP16",
    [preproc_input],
    outputs=[
        InferRequestedOutput("LOGITS"),
    ],
)

In [6]:
b = res.as_numpy("LOGITS")
print(b.shape)
print(b)

(3, 2)
[[1.9493872e-02 9.8050606e-01]
 [9.9910659e-01 8.9344563e-04]
 [9.9943739e-01 5.6257565e-04]]


In [7]:
triton_client = InferenceServerClient(url="localhost:8900")

pred_input = InferInput(
    name="LOGITS",
    shape=b.shape,
    datatype=trutils.np_to_triton_dtype(b.dtype),
)

pred_input.set_data_from_numpy(b)


res = triton_client.infer(
    "class-predictor",
    [pred_input],
    outputs=[
        InferRequestedOutput("PREDICTED_CLASS"),
    ],
)

In [8]:
c = res.as_numpy("PREDICTED_CLASS")
print(c)

[[1]
 [0]
 [0]]


In [9]:
image_paths = [
    "/images/cat1.jpg",
    "/images/cat2.jpg",
    "/images/cat3.jpg"
]
image_paths = np.array(image_paths, dtype=object)

triton_client = InferenceServerClient(url="localhost:8900")

input_images_path = InferInput(
    name="IMAGE_PATH",
    shape=image_paths.shape,
    datatype=trutils.np_to_triton_dtype(image_paths.dtype),
)

input_images_path.set_data_from_numpy(image_paths)


res = triton_client.infer(
    "ensemble-conv",
    [input_images_path],
    outputs=[
        InferRequestedOutput("PREDICTED_CLASS"),
    ],
)

In [10]:
res.as_numpy("PREDICTED_CLASS")

array([[1],
       [0],
       [0]], dtype=int32)